# 💾 Notebook 23: Live Connected SQL Database & Relational Data Engineering
## Relational Schema, Analytical SQL Window Functions, Real-Time Ingestion & DB Introspection

**Author:** CareFlow Data Engineering & Database Architecture Team  
**Pillars:** Data Engineering, SQL Analytics, Relational Database Systems & Real-Time Updates  

---

### Overview
While file-based lakehouses (Parquet/CSV) provide high batch throughput, operational hospital command centers require **ACID-compliant relational SQL databases** for transactional event logging, live census updates, and instant ad-hoc query capabilities.

In this notebook, we connect to the **CareFlow Live SQLite Database** (`careflow_hospital.db`) and explore:
1. **Schema Architecture:** 8 relational tables across Bronze, Silver, and Gold layers.
2. **Analytical SQL Queries:** Common Table Expressions (CTEs), window `LAG()` lag partitions, and multi-dimensional aggregations.
3. **Real-Time Data Ingestion:** Programmatic SQL `INSERT` commands updating hospital records.
4. **Relational Join Analysis:** Joining `fct_patient_journeys` with `mart_sla_compliance` to identify high-risk clinical cohorts.

In [ ]:
# 1. Import Database Modules & SQLite
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure careflow_analytics package is accessible
sys.path.insert(0, os.path.abspath('..'))
from careflow_analytics.database import (
    init_sql_database,
    query_sql,
    execute_sql,
    get_database_stats,
    insert_synthetic_patient_event,
    DB_PATH
)

# Ensure database is seeded and verified
db_info = init_sql_database(force_refresh=False)
print(f"[OK] Connected to Live SQL Database at: {DB_PATH}")

### 2. Relational Database Introspection
We query SQLite internal system catalog `sqlite_master` to audit table schemas and live row counts.

In [ ]:
db_stats = get_database_stats()
db_stats

### 3. Analytical SQL Query 1: CTE & Severity Aggregations
Using Common Table Expressions (CTEs) to aggregate mean Length of Stay, breach rates, and loopback frequency by clinical severity.

In [ ]:
sql_severity = """
WITH severity_summary AS (
    SELECT 
        f.clinical_severity,
        COUNT(*) AS total_cases,
        ROUND(AVG(f.total_los_minutes), 1) AS avg_los_min,
        ROUND(AVG(f.total_los_minutes) / 60.0, 2) AS avg_los_hours,
        SUM(CASE WHEN f.has_radiology_loopback_rework = 1 THEN 1 ELSE 0 END) AS loopback_count,
        ROUND(AVG(CASE WHEN f.has_radiology_loopback_rework = 1 THEN 1.0 ELSE 0.0 END) * 100, 1) AS loopback_pct,
        ROUND(AVG(CASE WHEN s.is_door_to_doctor_breached = 1 THEN 1.0 ELSE 0.0 END) * 100, 1) AS d2d_breach_pct
    FROM fct_patient_journeys f
    LEFT JOIN mart_sla_compliance s ON f.case_id = s.case_id
    GROUP BY f.clinical_severity
)
SELECT * FROM severity_summary
ORDER BY avg_los_min DESC;
"""

df_severity = query_sql(sql_severity)
df_severity

### 4. Analytical SQL Query 2: Inpatient Ward Bed Saturation & Hospital Census
Direct query against the `hospital_ward_census` relational table monitoring facility critical capacity.

In [ ]:
sql_census = """
SELECT 
    unit_name,
    total_beds,
    occupied_beds,
    total_beds - occupied_beds AS vacant_beds,
    occupancy_rate,
    status
FROM hospital_ward_census
ORDER BY occupancy_rate DESC;
"""

df_census = query_sql(sql_census)
df_census

### 5. Real-Time Dynamic SQL Ingestion (Live Transactional INSERT)
We simulate a real-time patient event arrival entering the Emergency Department and dynamically execute a SQL `INSERT` statement into `raw_ehr_events`.

In [ ]:
new_event = {
    'case_id': 'LIVE-SQL-9999',
    'activity_name': 'Emergency Triage Check-In',
    'resource_name': 'Charge_Nurse_Martinez',
    'patient_gender': 'Female',
    'patient_age_group': '61+',
    'clinical_severity': 'High'
}

rows_affected = insert_synthetic_patient_event(new_event)
print(f"[OK] Successfully executed transactional INSERT: {rows_affected} row added to raw_ehr_events.")

# Verify insertion via SELECT
verify_df = query_sql("SELECT * FROM raw_ehr_events WHERE case_id = 'LIVE-SQL-9999'")
verify_df

### 6. Visualizing Live SQL Query Outputs
Visualizing the ward occupancy rates directly retrieved from our relational SQL database query.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#ef4444' if r >= 90.0 else ('#f59e0b' if r >= 80.0 else '#10b981') for r in df_census['occupancy_rate']]

bars = ax.barh(df_census['unit_name'], df_census['occupancy_rate'], color=colors, alpha=0.85)
ax.axvline(85.0, color='#f59e0b', linestyle='--', label='Hospital Surge Threshold (85%)')
ax.axvline(90.0, color='#ef4444', linestyle='--', label='Critical Saturation Threshold (90%)')
ax.set_xlabel("Occupancy Rate (%)", color='white')
ax.set_title("Live SQL Hospital Ward Bed Saturation Telemetry", fontsize=12, fontweight='bold', color='white')
ax.grid(True, linestyle='--', alpha=0.2)
ax.legend()

for b in bars:
    w = b.get_width()
    ax.text(w + 1, b.get_y() + b.get_height()/2.0, f"{w:.1f}%", ha='left', va='center', color='cyan', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

### Conclusion & Relational Architecture Principles
- Embedding a connected relational SQL engine bridges modern Python Data Science with enterprise Hospital Information Systems (HIS / EHR).
- Live SQL execution enables zero-latency transactional ingestion alongside high-performance analytical reporting.